# RileysCrush on Google Colab

Runs the full RileysCrush app (Express API + built React client, single process) inside this Colab VM, then exposes it publicly with [localtunnel](https://github.com/localtunnel/localtunnel).

**Run the cells top to bottom.** The last cell prints a public `https://*.loca.lt` URL — that's the live site.

**Persistence note:** this VM's disk is wiped when the Colab runtime disconnects or recycles. `data/crushes.json` edits made through `/admin` during your session are real (unlike a Vercel serverless deployment) but won't survive a runtime restart unless you either (a) download `data/crushes.json` from the file browser afterwards, or (b) run the optional Google Drive cell below before starting the server.

## 1. Install Node.js 20

Colab's base image doesn't ship a recent-enough Node for Vite/this project (needs 18+), so install it via NodeSource.

In [ ]:
!curl -fsSL https://deb.nodesource.com/setup_20.x | bash -
!apt-get install -y nodejs
!node -v && npm -v

## 2. Clone the repo

In [ ]:
import os

REPO_URL = "https://github.com/DeQuackDealer/confess-web.git"
BRANCH = "claude/build-rileyscrush-f0276h"  # switch to "main" once this branch is merged
REPO_DIR = "/content/confess-web"

if not os.path.isdir(REPO_DIR):
    !git clone --branch {BRANCH} --single-branch {REPO_URL} {REPO_DIR}
else:
    print("Repo already present, pulling latest...")
    !cd {REPO_DIR} && git pull

## 3. Install dependencies and build the client

In [ ]:
%cd /content/confess-web
!npm install
!npm run build --workspace=client

## 4. Install localtunnel

In [ ]:
!npm install -g localtunnel

## 5. (Optional) Persist `data/` to Google Drive

Skip this cell if you don't care about hidden-message edits surviving a runtime restart. If you run it, `data/crushes.json` and `data/config.json` will live under `MyDrive/RileysCrush-data` instead of the VM's local disk.

In [ ]:
import os, shutil
from google.colab import drive

drive.mount('/content/drive')

DRIVE_DATA_DIR = '/content/drive/MyDrive/RileysCrush-data'
LOCAL_DATA_DIR = '/content/confess-web/data'

os.makedirs(DRIVE_DATA_DIR, exist_ok=True)

if not os.listdir(DRIVE_DATA_DIR):
    print("First run: seeding Drive with the repo's default data/ files.")
    for f in os.listdir(LOCAL_DATA_DIR):
        shutil.copy(os.path.join(LOCAL_DATA_DIR, f), DRIVE_DATA_DIR)

if os.path.islink(LOCAL_DATA_DIR):
    os.remove(LOCAL_DATA_DIR)
elif os.path.isdir(LOCAL_DATA_DIR):
    shutil.rmtree(LOCAL_DATA_DIR)

os.symlink(DRIVE_DATA_DIR, LOCAL_DATA_DIR)
print("data/ now points at Google Drive:", DRIVE_DATA_DIR)

## 6. Start the server and open a public tunnel

This starts the production server (serves the API + the built client on one port) and localtunnel in the background, waits for both to report ready, then prints the public URL.

In [ ]:
import subprocess, os, time, threading, queue, signal

PORT = 4000
env = os.environ.copy()
env["NODE_ENV"] = "production"
env["PORT"] = str(PORT)


def stop_previous_run():
    """Kills any server/tunnel left running from an earlier run of this cell in
    this same session, so re-running the cell (e.g. after `git pull`-ing a fix)
    doesn't fail with EADDRINUSE because the old server is still holding the
    port. `npm run start` spawns a shell that in turn spawns the actual
    tsx/node process — terminating just the top `npm` PID does NOT kill that
    grandchild, so each process is launched in its own session (start_new_session)
    and stopped by signalling the whole process group instead of just one PID."""
    for name in ("tunnel_proc", "server_proc"):
        proc = globals().get(name)
        if proc is not None and proc.poll() is None:
            try:
                os.killpg(os.getpgid(proc.pid), signal.SIGTERM)
            except ProcessLookupError:
                pass
            try:
                proc.wait(timeout=5)
            except subprocess.TimeoutExpired:
                try:
                    os.killpg(os.getpgid(proc.pid), signal.SIGKILL)
                except ProcessLookupError:
                    pass
                proc.wait(timeout=5)
            print(f"Stopped previous {name}.")


def start_process(cmd, cwd=None, env=None):
    return subprocess.Popen(
        cmd, cwd=cwd, env=env,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, bufsize=1,
        start_new_session=True,  # own process group, so stop_previous_run can kill the whole tree
    )


def pump_output(proc, label, q):
    """Continuously drains a process's stdout so it never blocks on a full pipe,
    prints each line with a label, and forwards it to a queue for the marker wait."""
    for line in proc.stdout:
        print(f"[{label}] {line}", end="")
        q.put(line)
    q.put(None)


def wait_for_marker(q, marker, timeout=60):
    deadline = time.time() + timeout
    while True:
        remaining = deadline - time.time()
        if remaining <= 0:
            return False
        try:
            line = q.get(timeout=remaining)
        except queue.Empty:
            return False
        if line is None:
            return False
        if marker in line:
            return True


stop_previous_run()

print("Starting RileysCrush server...")
server_q = queue.Queue()
server_proc = start_process(
    ["npm", "run", "start", "--workspace=server"],
    cwd="/content/confess-web", env=env,
)
threading.Thread(target=pump_output, args=(server_proc, "server", server_q), daemon=True).start()

if not wait_for_marker(server_q, "listening", timeout=60):
    print("\n⚠️ Server did not report 'listening' in time — check the log above for errors.")
else:
    print("\nStarting localtunnel...")
    tunnel_q = queue.Queue()
    tunnel_proc = start_process(["lt", "--port", str(PORT)])
    threading.Thread(target=pump_output, args=(tunnel_proc, "localtunnel", tunnel_q), daemon=True).start()

    if wait_for_marker(tunnel_q, "your url is", timeout=60):
        print("\n✅ RileysCrush is live at the URL printed above.")
        print("   First-time visitors may see localtunnel's interstitial page — click 'Click to Continue'.")
        print("   Try /studio and /admin on that same URL too.")
    else:
        print("\n⚠️ localtunnel did not report a URL in time — check the log above for errors.")

## 7. (Optional) Stop everything

Run this cell to shut down the server and tunnel cleanly (this reuses `stop_previous_run` from cell 6, so run that cell at least once first). Note cell 6 already does this automatically at the start of every run, so you only need this cell if you want to stop things *without* immediately starting a new run.

In [ ]:
stop_previous_run()